# AV2 + xHE-AAC on Google Colab

Same conversion as the desktop app. The video codec becomes AV2, the audio codec becomes xHE-AAC, and the source resolution, frame rate, and bit depth stay as they are.

**Runtime.** Free Colab’s strongest GPU is an **NVIDIA T4**. Choose **Runtime → Change runtime type → T4 GPU**, then run every cell from the top.

`avmenc` is the AV2 reference encoder and has no CUDA encode path, so the frames are encoded on the CPUs of that T4 machine. The first build of AVM takes a while. TensorFlow Lite is left out of this build because the small-file preset does not use the ML partition search.

**Input.** `sample/animation.mp4`

**Small-file preset.** `cpu-used=9` (fastest), `cq-level=55` (highest quantizer on the app’s quality control, so the smallest video), exhale mode `a` (lowest SBR preset). Mode `0` is smaller still, but exhale says that mode is only for when you must have the lowest rate.

In [ ]:
from pathlib import Path
import subprocess
import sys

def find_repo() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/converter"),
        Path("/content"),
    ]
    for path in candidates:
        video = path / "sample" / "animation.mp4"
        package = path / "src" / "av2converter"
        if video.is_file() and package.is_dir():
            return path.resolve()
    raise SystemExit(
        "Could not find sample/animation.mp4 next to src/av2converter. "
        "In Colab, upload the converter project (the sample folder included) "
        "and unzip it so /content/converter/sample/animation.mp4 exists, "
        "or open this notebook from the project directory."
    )

REPO = find_repo()
SAMPLE = REPO / "sample" / "animation.mp4"
print(f"Project: {REPO}")
print(f"Sample:  {SAMPLE} ({SAMPLE.stat().st_size / 1e6:.2f} MB)")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit(
        "No GPU is attached. Free Colab’s strongest GPU is the NVIDIA T4. "
        "Use Runtime → Change runtime type → T4 GPU, then run this cell again."
    )
name = gpu.stdout.strip()
print(f"GPU: {name}")
if "T4" not in name and "L4" not in name and "A100" not in name and "H100" not in name:
    print("This is not a T4. On the free tier, switch the runtime to a T4 if that option is listed.")
print("AV2 encoding uses the CPUs on this runtime. The T4 is the strongest free Colab GPU.")

In [ ]:
import shutil
import subprocess

subprocess.check_call(
    [
        "sudo", "apt-get", "update",
    ] if shutil.which("sudo") else ["apt-get", "update"]
)
install = [
    "apt-get", "install", "-y",
    "build-essential", "cmake", "nasm", "yasm", "git", "pkg-config",
    "python3", "ffmpeg",
]
if shutil.which("sudo"):
    install.insert(0, "sudo")
subprocess.check_call(install)
print("Build tools and ffmpeg are installed.")

In [ ]:
from pathlib import Path
import os
import subprocess

BUILD = Path("/content/av2-build") if Path("/content").exists() else REPO / ".colab-build"
BUILD.mkdir(parents=True, exist_ok=True)
BIN = BUILD / "bin"
BIN.mkdir(exist_ok=True)
os.environ["PATH"] = f"{BIN}:{os.environ.get('PATH', '')}"

avm_src = BUILD / "avm"
if not (BIN / "avmenc").exists():
    if not avm_src.exists():
        subprocess.check_call(
            [
                "git", "clone", "--depth", "1", "--branch", "v1.0.0",
                "https://github.com/AOMediaCodec/avm.git", str(avm_src),
            ]
        )
    avm_build = BUILD / "avm-build"
    avm_build.mkdir(exist_ok=True)
    subprocess.check_call(
        [
            "cmake", "-S", str(avm_src), "-B", str(avm_build),
            "-DCMAKE_BUILD_TYPE=Release",
            "-DBUILD_SHARED_LIBS=OFF",
            "-DENABLE_TESTS=OFF",
            "-DENABLE_DOCS=OFF",
            "-DENABLE_EXAMPLES=ON",
            "-DENABLE_TOOLS=OFF",
            "-DENABLE_NASM=ON",
            "-DCONFIG_ML_PART_SPLIT=0",
            "-DCONFIG_DIP_EXT_PRUNING=0",
            "-DCONFIG_TENSORFLOW_LITE=0",
        ]
    )
    subprocess.check_call(
        ["cmake", "--build", str(avm_build), "-j", str(os.cpu_count() or 2), "--target", "avmenc"]
    )
    built = next(avm_build.rglob("avmenc"))
    subprocess.check_call(["cp", "-a", str(built), str(BIN / "avmenc")])
print("avmenc:", BIN / "avmenc")

In [ ]:
import os
import subprocess

exhale_src = BUILD / "exhale"
if not (BIN / "exhale").exists():
    if not exhale_src.exists():
        subprocess.check_call(
            [
                "git", "clone", "--depth", "1",
                "https://gitlab.com/ecodis/exhale.git", str(exhale_src),
            ]
        )
    subprocess.check_call(["make", "-j", str(os.cpu_count() or 2), "release"], cwd=exhale_src)
    built = next((exhale_src / "bin").rglob("exhale"))
    subprocess.check_call(["cp", "-a", str(built), str(BIN / "exhale")])
print("exhale:", BIN / "exhale")

In [ ]:
import sys
sys.path.insert(0, str(REPO / "src"))

from av2converter.convert import ConvertSettings, convert_file

# Smallest file that stays inside the app's quality control.
# cq-level 55 is the highest (worst) quantizer the desktop slider allows.
# exhale mode "a" is the lowest SBR preset.
settings = ConvertSettings(cpu_used=9, cq_level=55, audio_mode="a")
output = REPO / "sample" / "animation.av2.mp4"

def log(line: str) -> None:
    print(line, flush=True)

dest = convert_file(SAMPLE, output=output, settings=settings, log=log)
src_size = SAMPLE.stat().st_size
out_size = dest.stat().st_size
print(f"Source: {src_size / 1e6:.2f} MB")
print(f"AV2:    {out_size / 1e6:.2f} MB  ({100 * out_size / src_size:.1f}% of source)")
print(f"Wrote {dest}")

In [ ]:
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    print(f"Not running in Colab. The file is at {output}")